# 37 · Capstone: the support desk as a production service with an eval gate and a drill
Level L6 · Part 6 · core · ~45 min · builds on 23, 28, 29, 30, 31, 32, 34, 35

Needs Docker with Compose for the kill drill (Step 9), about 300 MB of RAM. Without Docker the drill shows the recorded
run, labelled as recorded output, and the rest of the lesson still runs. The desk turn makes four model calls, replayed
from the recording, so no key is needed to run it.

**Before you start:** lessons 8, 16, 23, 28 and 34. The setup cell stops with `run lesson N first` if a file from one of
them is missing. The desk also wires in four more: 29 (traces and spans), 31 (guardrails and the audit log), 32
(retries and idempotency keys) and 35 (durable execution). Lesson 35's key rule is used here; you do not need its code.

## Goal
Ship the Brightside support desk as one service: a CI gate that blocks a bad release, a desk turn that is traced and
capped, a refund that goes through an idempotent service, and a kill drill on the Compose fleet. The drill kills a replica
under load: every lookup gets an answer, and the refund is charged once.

## Why it matters
Lessons 23 to 35 each made one part work. Put together as a notebook, the desk still fails in the ways those lessons fixed.
The cell below ships a candidate with no gate, using the golden-set runs saved by lesson 28.
**Success** is the share of golden tasks with no failing check. A task fails when its check list has at least one
reason in it. A release that drops success, or breaks a task that passed before, is a bad release.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../data/28").resolve()))  # lesson 28: the release gate and its saved golden-set runs
from release_gate import load, success  # noqa: E402

v1, v2 = load("v1"), load("v2")  # v1 is live; v2 is a candidate change


def failing(run: dict) -> set:
    return {t for t, reasons in run["checks"].items() if reasons}


print(f"ship v2 with no gate: success {success(v2):.0%} (live v1: {success(v1):.0%})")
print("golden tasks that pass on v1 and fail on v2:", len(failing(v2) - failing(v1)))

ship v2 with no gate: success 63% (live v1: 90%)
golden tasks that pass on v1 and fail on v2: 10


**What it makes easy:** one command decides ship or hold, and one drill shows the service survives a kill. **When you
don't need it:** one customer on one laptop. Then lessons 23 and 28 are enough.

## The idea
A production agent is a service with five parts around the model: a gate that decides what ships, a guard on what goes
in, a cap on what a turn may spend, a refund route that is safe to repeat, and a drill that proves the service survives a
failure. Each part is one of the earlier lessons. This lesson wires them together and checks each one with a real result.

Analogy: a commissioning test for a new plant. Each machine passed its own test. The commissioning run checks that they
still work together, with one machine switched off in the middle.

## Picture
```mermaid
flowchart LR
  T[Customer ticket] --> G[Input guard: redact]
  G --> A[Desk agent: model, capped]
  A -->|lookup or policy search| R[(Order data and policy text)]
  A -->|refund request| H[Senior-lead approval]
  H --> S[Refund route: idempotent key]
  S --> F[(Shared store: keys and charges)]
  A --> O[Traces and audit log]
  C[CI gate on saved runs] -.blocks a bad release.-> A
  F -.three replicas and a gateway in Compose.-> D[Kill drill]
```

## Requirements
Ready to ship when every line is true. Each names the lesson that taught the piece. Step 8 checks most of them in code.

- [ ] **Gated:** a candidate that drops golden tasks is blocked; the candidate that passes ships (28).
- [ ] **Traced:** one span per model call and per tool call; the customer's words are not in the traces (29).
- [ ] **Guarded:** card numbers and emails are redacted before the model; the desk offers no tool that writes to a customer (30, 31).
- [ ] **Accountable:** every refund has a named reviewer in the audit log (31).
- [ ] **Paid once:** a refund key is made once per request and repeats return the saved answer (32, 34).
- [ ] **Capped:** at most the step cap of model calls and the spend cap per turn (33; the caps are the app's settings).
- [ ] **Drilled:** under load, one replica killed: every lookup is answered within three tries, and the refund is charged once (34, 35).

## Build it
**Step 1 · the parts from earlier lessons (no call).** Two folders hold the parts. `data/23` has the desk's tools and
instructions (`desk.py`) and the customer memory (`memory.py`), both written by lesson 23. `apps/34` has the refund
service that the fleet runs. The golden set is the fictional world of lesson 08. The cell stops with a clear message if
lesson 23 has not run.

In [2]:
import atexit, json, os, re, shutil, subprocess, threading, time, uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import httpx, uvicorn
from google.genai import types  # the SDK's message types (importing them makes no call)
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.trace import Status, StatusCode

sys.path.insert(0, "..")
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

COURSE = Path("..").resolve()
for rel, lesson in {"data/23/desk.py": 23, "data/28/release_gate.py": 28, "apps/34/fleet/app.py": 34}.items():
    assert (COURSE / rel).exists(), f"run lesson {lesson} first: {rel} is missing"
sys.path.insert(0, str(COURSE / "data" / "23"))  # lesson 23 wrote desk.py and memory.py here
import desk  # lesson 23: order and policy tools, SYSTEM, TOOLS, CITED and an offline fallback
from desk import SENIOR_LIMIT_USD, SYSTEM, TOOLS, calls_in, fallback_turn, lookup_order, search_policy  # noqa: E402
from memory import facts_for  # lesson 16's customer memory, as lesson 23 used it  # noqa: E402

GOLDEN = json.loads((COURSE / "data" / "support_eval.json").read_text())
print(len(GOLDEN["tasks"]), "golden tasks ·", len(GOLDEN["world"]["orders"]), "fictional orders · order 1007:",
      GOLDEN["world"]["orders"]["1007"])

30 golden tasks · 12 fictional orders · order 1007: {'status': 'lost', 'days_late': 12, 'total': 210.0}


**Step 2 · the CI gate (no model, no Docker).** In CI the gate is a pytest run. The candidate's version comes from the
pull request. The test below asks lesson 28's `decide` whether the candidate may ship, then checks each hard task. A
failing test fails the build. The runs are saved, so the gate needs no model call.

In [3]:
%%writefile ../apps/37/ci/test_ship_gate.py
"""The CI gate of lesson 37. CI sets CANDIDATE to the version a pull request changes, then runs pytest:
a failing test fails the build. The rules and the saved golden-set runs are lesson 28's."""
import json
import os
import sys
from pathlib import Path

import pytest

DATA28 = Path(__file__).resolve().parents[3] / "data" / "28"  # apps/37/ci -> the course folder -> data/28
sys.path.insert(0, str(DATA28))
from release_gate import decide, load  # noqa: E402  (lesson 28)

RULES = json.loads((DATA28 / "release_gate.json").read_text())
CANDIDATE = (Path(__file__).parent / "candidate.txt").read_text().strip()  # CI writes the version the pull request changes


def test_candidate_has_a_saved_run():
    assert load(CANDIDATE)["version"] == CANDIDATE


def test_candidate_passes_the_release_gate():
    ship, reasons = decide(load("v1"), load(CANDIDATE), RULES)
    assert ship, "blocked: " + "; ".join(reasons)


@pytest.mark.parametrize("task", RULES["must_pass"])  # one line per hard task in the report
def test_hard_task_passes(task):
    assert not load(CANDIDATE)["checks"][task], load(CANDIDATE)["checks"][task]

Overwriting ../apps/37/ci/test_ship_gate.py


**Run the gate (no model).** The same tests run once per candidate, as CI would run them. The gate's verdict is the
exit code of pytest. The candidate's version is written to `candidate.txt`, the file CI writes.

In [4]:
GATE_RESULT = {}  # candidate -> True when the gate ships it
CI_DIR = COURSE / "apps" / "37" / "ci"


def run_gate(candidate: str) -> tuple[int, str]:  # what CI does: write the candidate, run the tests
    (CI_DIR / "candidate.txt").write_text(candidate + "\n")
    done = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(CI_DIR)],
                          capture_output=True, text=True)
    return done.returncode, done.stdout.strip().splitlines()[-1]


from release_gate import decide  # lesson 28: the same rules the test reads
RULES = json.loads((COURSE / "data" / "28" / "release_gate.json").read_text())
for candidate in ("v2", "v3"):
    code, last = run_gate(candidate)
    GATE_RESULT[candidate] = code == 0
    print(candidate, "→", "SHIP" if code == 0 else "BLOCKED", "·", last)
    print("  decide's reasons:", decide(load("v1"), load(candidate), RULES)[1] or "none")

v2 → BLOCKED · 3 failed, 2 passed in 0.07s
  decide's reasons: ['success fell from 90% to 63%', "passed before, fails now: ['t09', 't12', 't14', 't15', 't17', 't23', 't25', 't26', 't27', 't29']", "hard tasks failed: ['t09', 't27']"]


v3 → SHIP · 5 passed in 0.02s
  decide's reasons: none


**In a real CI (quoted sketch, not run here).** The same test runs on every pull request. The YAML below is the shape of
the workflow file, not a file this lesson runs. Its paths assume the course folder is the repository root. The `v2` and
`v3` runs in `data/28/runs` are prepared files from lesson 28, not the output of a pull request. A real CI must make one
run per pull request: the author records the candidate's golden-set run (lesson 28, the model call happens before the
pull request) and commits it, and CI reads which run file the pull request changed.

```yaml
name: ship-gate
on: pull_request
jobs:
  gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0                 # the diff below needs origin/main
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -q pytest
      - name: Write the candidate from the pull request
        run: basename "$(git diff --name-only origin/main -- data/28/runs/ | head -1)" .json > apps/37/ci/candidate.txt
      - name: Gate
        run: pytest apps/37/ci
```

**Step 3 · the refund service (no model).** The refund route is lesson 34's service: the same `app.py` and `store.py` the
containers run. Here one replica runs in this process on this lesson's port, with the keys kept in memory. A refund needs
an `Idempotency-Key` header. The first try moves the money; a repeat with the same key gets the saved answer. The gateway
(lesson 34, nginx) spreads lookups over the replicas, and Redis holds the state they share: refund keys and the charge
count. Here there is no Redis, so one process keeps both in memory.

In [5]:
FLEET = COURSE / "apps" / "34" / "fleet"
os.environ.update(ORDERS_FILE=str(COURSE / "data" / "support_eval.json"), AGENT_VERSION="v1",
                  REPLICA_NAME="desk-local", REDIS_URL="")  # no Redis: one process keeps the keys in memory
sys.path[:0] = [str(FLEET), str(COURSE / "apps" / "34")]
import app as refund_app  # noqa: E402  (lesson 34): /lookup, /refunds with Idempotency-Key, /charges
import fleet_tools  # noqa: E402  (lesson 34): wait_for, load_test, Compose helpers

REFUND_SERVICE = "http://127.0.0.1:8370"  # this lesson's port (progress.py --ports 37)
server = uvicorn.Server(uvicorn.Config(refund_app.app, host="127.0.0.1", port=8370, log_level="warning"))
replica = threading.Thread(target=server.run, daemon=True)
replica.start()
atexit.register(lambda: setattr(server, "should_exit", True))
print("refund service:", fleet_tools.wait_for(f"{REFUND_SERVICE}/health"))

refund service: {'ok': True, 'version': 'v1'}


**Step 4 · the guards and the tool list (no model).** Two rules from lessons 30 and 31. The input guard redacts a card
number and an email before the model sees the ticket. The desk offers the model three tools only: it cannot send a
message to the customer, so a planted instruction in a ticket has no tool to call.

In [6]:
CARD = re.compile(r"\b\d(?:[ -]?\d){12,15}\b")  # 13 to 16 digits, with spaces or dashes
EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")


def redact(text: str) -> str:  # the input guard: runs before the model and before any log
    return EMAIL.sub("[email]", CARD.sub("[card]", text))


TICKET = "Order 1007 never came. Please refund it. My card is 4111-1111-1111-1112 and my email is name@example.com."
print(redact(TICKET))
DECLARED = [d["name"] for d in TOOLS[0]["function_declarations"]]
print("tools the desk offers:", DECLARED)
assert "send_reply" not in DECLARED, "least privilege: the desk must not write to the customer"

Order 1007 never came. Please refund it. My card is [card] and my email is [email].
tools the desk offers: ['lookup_order', 'search_policy', 'refund_order']


**Step 5 · the caps (no model).** A turn stops after the step cap of model calls, or when its spend reaches the spend
cap. The caps are the app's own settings, read from `.env` with the defaults shown. The cost of a reply is its tokens
times the model's price (lesson 33). The function is checked before any call is made.

In [7]:
MODEL = env("TUT_LLM_MODEL", "gemini-3.1-flash-lite")  # the course's default model
PRICE_IN, PRICE_OUT = 0.25, 1.50  # USD per 1M tokens, as checked for lesson 33 on 2026-10-20
MAX_STEPS = int(env("SUPPORT_MAX_STEPS", "4"))  # model calls per turn
MAX_USD = float(env("SUPPORT_MAX_USD", "0.01"))  # USD per turn
CALLS = {"n": 0, "usd": 0.0}  # real model calls and their cost in this kernel


def budget_ok(steps: int, spent: float) -> str | None:  # None means the turn may make another call
    if steps >= MAX_STEPS:
        return f"step cap of {MAX_STEPS} model calls reached"
    if spent >= MAX_USD:
        return f"spend cap of {MAX_USD} USD reached"
    return None


print("caps:", MAX_STEPS, "calls and", MAX_USD, "USD per turn")
print(budget_ok(2, 0.002), "|", budget_ok(4, 0.0), "|", budget_ok(1, 0.02))

caps: 4 calls and 0.01 USD per turn
None | step cap of 4 model calls reached | spend cap of 0.01 USD reached


**Step 6 · traces and the audit log (no model).** Every model call and tool call is a span, written as one JSON line.
The spans carry counts, names and the model, never the customer's words (lesson 29). The audit log records who did what
on whose behalf, and the reviewer of each refund (lesson 31).

In [8]:
DATA37 = COURSE / "data" / "37"
DATA37.mkdir(parents=True, exist_ok=True)
TRACE_FILE, AUDIT_FILE = DATA37 / "traces.jsonl", DATA37 / "audit.jsonl"
for f in (TRACE_FILE, AUDIT_FILE):  # a clean run
    f.unlink(missing_ok=True)
provider = TracerProvider()
exporter = ConsoleSpanExporter(out=TRACE_FILE.open("w"), formatter=lambda span: span.to_json(indent=None) + "\n")
provider.add_span_processor(SimpleSpanProcessor(exporter))  # each span is written as soon as it ends
tracer = provider.get_tracer("brightside-support-desk")


def audit(**row) -> None:  # who did what: no customer text, no card number, no email
    with AUDIT_FILE.open("a") as f:
        f.write(json.dumps({"at": datetime.now(timezone.utc).isoformat(timespec="seconds"), **row}) + "\n")


print("traces go to", TRACE_FILE.name, "· audit goes to", AUDIT_FILE.name)

traces go to traces.jsonl · audit goes to audit.jsonl


**Step 7 · the desk turn: the tools, the approval and the loop (no model call here).** A refund over the senior limit
needs a senior-lead decision. The decision is typed in as data here, as in lesson 23. The refund goes to the service with
a key made from the run and the order, so a retry of the same request sends the same key (lesson 35). Lookups and the
policy search run in this process. `client` is a module-level name: Step 7b sets it to the model client, and `ask_model`
reads it. While it is still `None`, the offline fallback of Step 7c answers instead. That one name is the whole switch.
In replay the refund request's answer comes from the recording, so this process's service may show no charge. The
Step 8 check reads `REFUND_LOG`, which holds the answers the desk received, recorded or live.

In [9]:
THREAD = "desk-C-88-1007"  # one customer request = one run id
QUESTION = ("I want a refund for order 1007, it never arrived. Please refund the 210.00 USD I paid. "
            "What does your policy say about a lost parcel?")
REVIEW = {"reviewer": "senior-lead", "approve": True}  # the person's decision (lesson 14 and 23)
REFUND_LOG = []  # every answer the refund service gave the desk, in order (a replay returns the recorded answer)
LOCAL_TOOLS = {"lookup_order": lookup_order, "search_policy": search_policy}
client = None  # set in Step 7b; while it is None the offline fallback answers


def refund_via_service(args: dict, thread: str) -> dict:
    order, amount = args["order_id"], float(args["amount_usd"])
    print("approval asked:", {"order_id": order, "amount_usd": amount}, "· needs senior-lead:", amount > SENIOR_LIMIT_USD)
    if amount > SENIOR_LIMIT_USD and REVIEW["reviewer"] != "senior-lead":
        return {"status": "held: a refund over the limit needs senior-lead"}
    if not REVIEW["approve"]:
        return {"status": f"rejected by {REVIEW['reviewer']}"}
    key = f"refund:{thread}:{order}"  # made from the run, not the attempt: every retry sends the same key
    r = httpx.post(f"{REFUND_SERVICE}/refunds", json={"order_id": order, "amount_usd": amount},
                   headers={"Idempotency-Key": key}, timeout=5)
    r.raise_for_status()
    REFUND_LOG.append(r.json())
    audit(actor="support-desk", on_behalf_of=thread, tool="refund_order", reviewer=REVIEW["reviewer"],
          decision="paid", amount_usd=amount)
    return {"status": "refunded", "amount_usd": amount, "first_try": r.json()["first_try"]}

**Run a tool.** A refund goes to the service. A lookup or a policy search runs here. Either way the call is one span
and one audit line.

In [10]:
def run_tool(call, thread: str):
    with tracer.start_as_current_span(f"execute_tool {call.name}") as span:  # the arguments are not logged
        span.set_attribute("gen_ai.tool.name", call.name)
        args = dict(call.args)
        if call.name == "refund_order":
            result = refund_via_service(args, thread)
        else:
            result = LOCAL_TOOLS[call.name](**args)
            audit(actor="support-desk", on_behalf_of=thread, tool=call.name, decision="read")
    return types.Part.from_function_response(name=call.name, response=result)

The model call and the loop. Each model call is one `chat` span. The loop stops at the caps from Step 5, and a reply
with no tool request is the answer.

In [11]:
def ask_model(history: list, facts: list[str]):
    if client is None:  # the model cell was skipped: labelled offline steps, never model output
        return fallback_turn(history), 0.0
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM + ("\nKnown facts about this customer: " + "; ".join(facts) if facts else ""),
        tools=TOOLS, temperature=0, max_output_tokens=512)
    with tracer.start_as_current_span(f"chat {MODEL}") as span:  # one model call = one chat span
        reply = client.models.generate_content(model=MODEL, contents=history, config=config)
        u = reply.usage_metadata
        prompt, out = u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
        span.set_attributes({"gen_ai.operation.name": "chat", "gen_ai.request.model": MODEL,
                             "gen_ai.usage.input_tokens": prompt, "gen_ai.usage.output_tokens": out})
    CALLS["n"] += 1
    usd = (prompt * PRICE_IN + out * PRICE_OUT) / 1_000_000
    CALLS["usd"] += usd
    return reply.candidates[0].content, usd

**The turn.** The customer's text is redacted, then the loop runs under the caps until the model answers.

In [12]:
def desk_turn(question: str, thread: str) -> str:
    facts = facts_for("C-88")  # lesson 16: the saved facts about this customer
    history = [types.Content(role="user", parts=[types.Part(text=redact(question))])]  # the input guard runs first
    spent, steps = 0.0, 0
    with tracer.start_as_current_span("invoke_agent support-desk") as agent:
        agent.set_attribute("gen_ai.conversation.id", thread)
        while (stop := budget_ok(steps, spent)) is None:
            turn, usd = ask_model(history, facts)
            steps, spent = steps + 1, spent + usd
            calls = calls_in(turn)
            if not calls:  # no tool request: the model has answered
                return "".join(p.text or "" for p in turn.parts)
            history.append(turn)
            history.append(types.Content(role="user", parts=[run_tool(c, thread) for c in calls]))
        agent.set_status(Status(StatusCode.ERROR, stop))
    return f"stopped: {stop}"

**Step 7b · the model cell (needs-llm, four model calls).** The recorder covers this cell only. The turn makes at most
four model calls (the step cap, and the lesson's budget). The recording holds the four it makes: two policy searches (the
model asked for "lost parcel", then "refund"), the refund request and the answer. The refund itself is one request to the
service, made in this cell and recorded with the calls.

In [13]:
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("37_capstone-support-desk")  # the recorder covers this cell only

from google import genai  # the official Google SDK

client = genai.Client(vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"))
answer = desk_turn(QUESTION, THREAD)
llm_replay.stop()  # saves the recording (record mode) or closes the replay
print("answer:", answer)
print("model calls:", CALLS["n"], "of", MAX_STEPS, "· cost:", round(CALLS["usd"], 5), "USD")

LLM mode: replay (37_capstone-support-desk.yaml)


approval asked: {'order_id': '1007', 'amount_usd': 210.0} · needs senior-lead: True


answer: Hello,

I am sorry to hear that your order 1007 never arrived.

Regarding our policy on lost parcels, our records indicate that if we are unable to fulfill an order, we are required to provide a prompt refund for the total amount paid [policy-27, policy-31].

I have processed your request for a refund of 210.00 USD. Please note that a refund requires approval from a member of our team; the current status of your refund request is "refunded."

We will send a confirmation to your email address once the process is complete.
model calls: 4 of 4 · cost: 0.00069 USD


**Step 7c · offline fallback (runs only if Step 7b was skipped, that is, if no `answer` exists; the steps are typed in, not
model output).** The desk's fixed steps from lesson 23 answer instead. The refund still goes through the service, so the
checks below still run.

In [14]:
if "answer" not in globals():
    client = None  # never call the model from here: the fallback takes over
    answer = desk_turn(QUESTION, THREAD)
    print(answer)

**Step 8 · check the turn in code (no model).** Each line is one requirement from the list above. The checks read the
audit log, the traces, the refund answers the desk received and the caps. They make no model call. The service's own
charge count is checked by the drill in Step 9, where the service runs for real. `desk.CITED` is the set of passage ids
that the policy search has returned to the desk so far; the answer may cite only those.

In [15]:
rows = [json.loads(line) for line in AUDIT_FILE.read_text().splitlines()]
spans = [json.loads(line) for line in TRACE_FILE.read_text().splitlines()]
chat_spans = [s for s in spans if s["name"].startswith("chat ")]
cited = re.findall(r"policy-(\d+)", answer)
checks = {
    "policy searched": any(r["tool"] == "search_policy" for r in rows),
    "answer cites only passages the search showed": bool(cited) and all(int(i) in desk.CITED for i in cited),
    "refund approved by senior-lead and paid once": any(r.get("reviewer") == "senior-lead" and r["decision"] == "paid"
                                                       for r in rows) and [x["first_try"] for x in REFUND_LOG] == [True],
    "within the caps": CALLS["n"] <= MAX_STEPS and CALLS["usd"] < MAX_USD,
    "one chat span per model call": len(chat_spans) == CALLS["n"],
    "no customer text in the traces": QUESTION not in TRACE_FILE.read_text(),
    "no tool that writes to the customer": "send_reply" not in DECLARED,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", name)
print("passed:", sum(checks.values()), "of", len(checks))
print("audit, who did what:")
for r in rows:
    print(" ", r["actor"], "·", r["tool"], "· on behalf of", r["on_behalf_of"], "·", r.get("reviewer", "-"), "·", r["decision"])

PASS policy searched
PASS answer cites only passages the search showed
PASS refund approved by senior-lead and paid once
PASS within the caps
PASS one chat span per model call
PASS no customer text in the traces
PASS no tool that writes to the customer
passed: 7 of 7
audit, who did what:
  support-desk · search_policy · on behalf of desk-C-88-1007 · - · read
  support-desk · search_policy · on behalf of desk-C-88-1007 · - · read
  support-desk · refund_order · on behalf of desk-C-88-1007 · senior-lead · paid


**Step 9 · the Compose fleet and the kill drill (needs-docker).** The fleet is lesson 34's Compose file, under this
lesson's project name and gateway port: three v1 replicas, a gateway and Redis. The start cell waits until the gateway
answers. The gateway retries a failed `GET` on another replica, but only before the reply has started. A reply cut off
mid-body can reach the client, and a `GET` is safe to send again. A refund is never sent again without its key. The
Temporal worker drill of lesson 35 is not run again here: the same key rule covers it.

> **To run the drill live**
> - Set the switch in the shell before you start Jupyter, with both words, a comma and no spaces, as in lesson 34:
>   `export TUT_ENABLE=docker,slow`, then `jupyter lab` from the same shell. The switch is read from the real environment,
>   not from `.env`.
> - The first start builds the image. The course did not record a build time, so measure yours; the cell waits for it.
> - Port 8371 must be free. Check with `ss -ltn | grep 8371` (no output means free).
> - Without the switch, the drill cells print "Skipped" and show the recorded output, labelled as recorded.

In [16]:
HAS_DOCKER = bool(shutil.which("docker")) and subprocess.run(
    ["docker", "compose", "version"], capture_output=True).returncode == 0
STACK, GATEWAY_PORT = "tut-agentic-ai-15h-37", 8371  # this lesson's project and gateway port (lesson 34 uses 8340)
GATEWAY = f"http://localhost:{GATEWAY_PORT}"
fleet_tools.GATEWAY = GATEWAY  # lesson 34's helpers read this module-level address
RECORDED_FILE = COURSE / "data" / "recorded" / "37.json"  # a real run, shipped with the course
RECORDED = json.loads(RECORDED_FILE.read_text()) if RECORDED_FILE.exists() else {}
RESULTS, FLEET_UP = {}, False
print("docker with compose:", HAS_DOCKER, "· recorded results shipped for:", sorted(RECORDED))


def compose(*args: str, timeout: int = 600) -> str:  # this project only; the port comes from this lesson
    done = subprocess.run(["docker", "compose", "-p", STACK, "-f", "../infra/serving/compose.yml", *args],
                          capture_output=True, text=True, timeout=timeout,
                          env={**os.environ, "TUT_GATEWAY_PORT": str(GATEWAY_PORT)})
    if done.returncode:
        raise RuntimeError(done.stderr[-800:])
    return done.stdout


def show(key: str) -> None:  # a live result, or the recorded one, labelled
    print(RESULTS[key] if key in RESULTS else "recorded output (the fleet did not run here): " + str(RECORDED.get(key)))

docker with compose: True · recorded results shipped for: ['before_kill', 'drill_passed', 'killed', 'load', 'refund', 'replicas_after', 'up']


**Start the fleet (needs-docker, slow).** The first start builds the image, so it is the slow step. Then warm-up lookups
repeat until all three replicas answer. The next cell prints what came up.

In [17]:
if HAS_DOCKER:
    compose("up", "-d", "--build", "--wait", "--scale", "agent=3", "redis", "agent", "gateway")
    RESULTS["up"] = {"gateway": fleet_tools.wait_for(f"{GATEWAY}/health", timeout=120)["ok"],
                     "replicas": len(compose("ps", "-q", "agent").split())}
    FLEET_UP = True
    deadline = time.monotonic() + 60  # the gateway learns each new replica within about 2 s: poll, do not sleep
    while True:  # warm-up: repeat until all three replicas answer, so the drill starts from a full pool
        RESULTS["before_kill"] = fleet_tools.load_test(n=300)
        if RESULTS["before_kill"]["replicas"] == 3 or time.monotonic() > deadline:
            break

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [18]:
show("up")

recorded output (the fleet did not run here): {'gateway': True, 'replicas': 3}


**The drill (needs-docker).** A warm-up shows the three replicas answering. Then 600 lookups run, and the first replica
is killed with SIGKILL 0.2 s in: no shutdown and no cleanup. Once it is gone, the same refund is sent three times through
the gateway, each with the same key. The drill passes when every lookup is answered within three tries (this count is
reported, not required to be above 0), the refund is answered by a live replica, one try moves the money, and the charge
count is 1. The count of lookups that needed a second try is printed. The fleet is torn down in the `finally` block,
even if a step fails.

**If the drill fails:** (1) run `docker ps`: are the three agent replicas and the gateway running? (2) free port 8371, then
check it with `ss -ltn | grep 8371`. (3) Run the cell again. The last 200 log lines of the fleet are saved to
`data/37/fleet.log` before the teardown wipes the containers.

The helpers for the drill. First the kill, and lookups that are sent again when no whole reply came back.

In [19]:
def kill_one_replica() -> str:
    first = compose("ps", "-q", "agent").split()[0]  # the first v1 replica
    subprocess.run(["docker", "kill", "--signal", "KILL", first], check=True, capture_output=True)
    return first[:12]


def lookups_with_retry(n: int = 600, workers: int = 20, tries: int = 3) -> dict:  # n lookups through the gateway
    def one(i: int) -> dict:  # a GET is safe to send again: a reply cut off mid-body is asked for once more
        order = fleet_tools.ORDER_IDS[i % len(fleet_tools.ORDER_IDS)]
        for attempt in range(1, tries + 1):
            try:
                if fleet_tools.CLIENT.get(f"{GATEWAY}/lookup/{order}").status_code == 200:
                    return {"code": 200, "attempts": attempt}
            except httpx.HTTPError:
                pass  # no whole reply arrived: ask again
        return {"code": 0, "attempts": tries}

    with ThreadPoolExecutor(max_workers=workers) as pool:
        rows = list(pool.map(one, range(n)))
    return {"requests": n, "codes": dict(Counter(str(r["code"]) for r in rows)),
            "needed_a_second_try": sum(r["attempts"] > 1 for r in rows)}

A refund is sent again with the same key. The drill makes one key and sends it on every try, so the service can tell
the tries apart. `stop_fleet` is here too: it keeps the last logs, then removes the fleet.

In [20]:
def refund_with_retries(order_id: str, amount_usd: float, key: str, tries: int = 4) -> dict:
    for attempt in range(1, tries + 1):
        try:
            r = fleet_tools.CLIENT.post(f"{GATEWAY}/refunds", json={"order_id": order_id, "amount_usd": amount_usd},
                                        headers={"Idempotency-Key": key}, timeout=3)
            if r.status_code == 200:
                return {"attempt": attempt, **r.json()}
        except httpx.HTTPError:
            pass  # the reply was lost: send the same key again
    return {"attempt": None}


def stop_fleet() -> None:
    try:  # the logs go first: the teardown removes the containers
        (DATA37 / "fleet.log").write_text(compose("logs", "--no-color", "--tail", "200"))
    except RuntimeError:
        pass  # no logs to keep: the teardown still runs
    compose("down", "-v", "--remove-orphans")  # the fleet, its network and its volumes go

The drill itself. The `finally` block saves the fleet's last log lines, tears the fleet down and saves the results, even if
a step fails.

In [21]:
if FLEET_UP:
    try:
        load = {}
        loader = threading.Thread(target=lambda: load.update(lookups_with_retry(n=600)))  # 600 lookups, 20 at a time
        gone = threading.Event()
        killer = threading.Timer(0.2, lambda: (RESULTS.update(killed=kill_one_replica()), gone.set()))  # 0.2 s in
        loader.start()
        killer.start()
        gone.wait(timeout=60)  # the refunds start once the replica is gone
        tries = [refund_with_retries("1011", 15.0, "refund:drill-1011") for _ in range(3)]
        loader.join()
        killer.join()
        RESULTS["load"] = load
        RESULTS["refund"] = {"tries": tries, "charges": fleet_tools.CLIENT.get(f"{GATEWAY}/charges").json()["charges"]}
        RESULTS["replicas_after"] = compose("ps", "--format", "{{.Service}} {{.State}}").splitlines()
        RESULTS["drill_passed"] = (load["codes"] == {"200": 600} and RESULTS["refund"]["charges"] == 1
                                   and sum(t.get("first_try") is True for t in tries) == 1
                                   and RESULTS["before_kill"]["replicas"] == 3
                                   and all(t.get("replica") != RESULTS["killed"] for t in tries))
    finally:
        stop_fleet()
        (DATA37 / "run.json").write_text(json.dumps(RESULTS, indent=2, default=str))
        print("fleet stopped; results saved to data/37/run.json")

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Result.** The drill's numbers: live when Docker ran here, otherwise the recorded run, labelled as such. In the recorded
run no lookup needed a second try (`needed_a_second_try: 0`): the kill landed between replies. The retry is there for the
case when it does not, and it costs nothing when unused.

In [22]:
for key in ("before_kill", "killed", "load", "refund", "replicas_after", "drill_passed"):  # live if Docker ran here, else recorded
    show(key)

recorded output (the fleet did not run here): {'requests': 300, 'codes': {'200': 300}, 'p50_ms': 14.2, 'p95_ms': 63.0, 'replicas': 3, 'versions': {'v1': 300}, 'one_shared_counter': True}
recorded output (the fleet did not run here): dc98c38d7aae
recorded output (the fleet did not run here): {'requests': 600, 'codes': {'200': 600}, 'needed_a_second_try': 0}
recorded output (the fleet did not run here): {'tries': [{'attempt': 1, 'first_try': True, 'order_id': '1011', 'amount_usd': 15.0, 'replica': '4dda41817113'}, {'attempt': 1, 'first_try': False, 'order_id': '1011', 'amount_usd': 15.0, 'replica': '4dda41817113'}, {'attempt': 1, 'first_try': False, 'order_id': '1011', 'amount_usd': 15.0, 'replica': '4dda41817113'}], 'charges': 1}
recorded output (the fleet did not run here): ['agent running', 'agent running', 'gateway running', 'redis running']
recorded output (the fleet did not run here): True


**Decide.** The release is ship only when every check in Step 8 holds, the gate shipped the candidate, and the drill
passed (live or recorded).

In [23]:
drill = RESULTS.get("drill_passed", RECORDED.get("drill_passed", False))
decision = all(checks.values()) and GATE_RESULT["v3"] and bool(drill)
print("gate v3:", GATE_RESULT["v3"], "· checks:", sum(checks.values()), "of", len(checks), "· drill:", bool(drill))
print("decision:", "ship" if decision else "hold")

gate v3: True · checks: 7 of 7 · drill: True
decision: ship


## Reference solution
The cell below is collapsed. It maps each requirement to the check that proves it, for the run above.

**Answer** — try it first, then expand the hidden cell below.

In [24]:
mapping = {
    "Gated": f"v2 blocked, v3 shipped (Step 2): {GATE_RESULT}",
    "Traced": "chat spans per model call and no customer text in the traces (Step 8)",
    "Guarded": "redaction (Step 4) and the tool list without a send tool (Step 8)",
    "Accountable": "senior-lead named on the refund row of the audit log (Step 8)",
    "Paid once": f"the service's answers to the desk: {[x['first_try'] for x in REFUND_LOG]} (Step 8); drill charges: {RESULTS.get('refund', RECORDED.get('refund', {})).get('charges')}",
    "Capped": f"model calls {CALLS['n']} of {MAX_STEPS}; spend {round(CALLS['usd'], 5)} of {MAX_USD} USD (Steps 5 and 8)",
    "Drilled": f"drill passed: {drill} (Step 9; live here, or the recorded run when Docker did not run)",
}
for requirement, proof in mapping.items():
    print(f"{requirement:<12} → {proof}")

Gated        → v2 blocked, v3 shipped (Step 2): {'v2': False, 'v3': True}
Traced       → chat spans per model call and no customer text in the traces (Step 8)
Guarded      → redaction (Step 4) and the tool list without a send tool (Step 8)
Accountable  → senior-lead named on the refund row of the audit log (Step 8)
Paid once    → the service's answers to the desk: [True] (Step 8); drill charges: 1
Capped       → model calls 4 of 4; spend 0.00069 of 0.01 USD (Steps 5 and 8)
Drilled      → drill passed: True (Step 9; live here, or the recorded run when Docker did not run)


## At real scale
- **The gate reads a run made for each change.** Here the candidate's run is saved. CI makes that run on every pull
  request, and the gate also reads cost and latency (lessons 28 and 33).
- **Money moves in one place.** The refund route owns the key, and every replica sends the same key for the same
  request (lessons 32 and 34). A run that crosses deploys needs lesson 35's durable record.
- **Traces and audit logs have readers who did not write the agent.** Keep customer text out (lessons 29, 31).

- **Drills run on purpose.** Kill a replica in staging on a schedule. Regions are lesson 36.
- **Retry only what is safe to repeat.** A `GET` may be sent again; a refund only with its key (lessons 32, 34).
- **Retry at the client, not only at the gateway.** nginx retries on another replica only before it has started sending
  the reply (`proxy_next_upstream`). A reply cut off mid-body is not retried by the gateway, so the client's `GET` retry
  is needed (lesson 34).
- **Compose is one machine.** Kubernetes is its own course.

## Common mistakes
1. **A new key on every try.** A key made from a random id is new on each try, so the service charges twice. The key
   must be made once, from the run and the order, before the first try.

In [25]:
keys = [f"refund:{uuid.uuid4()}" for _ in range(2)]  # WRONG: a new key on each try
assert keys[0] == keys[1], "two tries, two keys: the service cannot tell they are one refund"

AssertionError: two tries, two keys: the service cannot tell they are one refund

The fix is the line in `refund_via_service`: `key = f"refund:{thread}:{order}"`, made once per request.

## Try it
Send a first try for 50.00 USD with the key `refund:try-it:1007`. Then send a second try with the same key for 20.00 USD.
Predict first: what amount comes back for the second try, and how many charges does it add? Use `httpx.post` to
`REFUND_SERVICE` with an `Idempotency-Key` header, as `refund_via_service` does. This exercise uses its own key, so it does
not depend on the desk's refund in Step 7 (in a replay that refund is answered from the recording). No model is called.

In [26]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [27]:
KEY = "refund:try-it:1007"  # a key of this exercise, not the desk's
first = httpx.post(f"{REFUND_SERVICE}/refunds", json={"order_id": "1007", "amount_usd": 50.0},
                   headers={"Idempotency-Key": KEY}, timeout=5).json()
before = httpx.get(f"{REFUND_SERVICE}/charges").json()["charges"]
second = httpx.post(f"{REFUND_SERVICE}/refunds", json={"order_id": "1007", "amount_usd": 20.0},
                    headers={"Idempotency-Key": KEY}, timeout=5).json()
after = httpx.get(f"{REFUND_SERVICE}/charges").json()["charges"]
print("first try:", first)
print("second try, same key, 20.00 USD:", second)
print("charges added by the second try:", after - before)

first try: {'first_try': True, 'order_id': '1007', 'amount_usd': 50.0, 'replica': 'desk-local'}
second try, same key, 20.00 USD: {'first_try': False, 'order_id': '1007', 'amount_usd': 50.0, 'replica': 'desk-local'}
charges added by the second try: 0


## Recap
- One gate decides the release from saved runs: the candidate that drops golden tasks is blocked.
- The desk turn is traced, guarded and capped, and its refund goes through one idempotent route with a key per request.
- The drill kills a replica under load: every lookup is answered within three tries, and the refund is charged once.
- The four model calls are recorded once, within the budget. Every other check in this lesson runs with no model call.

**Interview:** "How would you ship a support agent that can refund money?" Strong answer: "Gate every release on a saved
golden set in CI, trace and cap each turn, and route refunds through one service with an idempotency key per request.
Name a human reviewer for each refund, and drill a replica kill before you go live."

**Stop the refund service.** The last code cell, so the cells above could still reach it.

In [28]:
server.should_exit = True
replica.join(timeout=10)

## Key terms
- **Capstone** — a lesson that puts the parts of a course into one working service and checks it with real results
  (lesson 37).
- **Release gate in CI** — a test that reads saved runs and fails the build when a candidate breaks a rule (lesson 28).
- **Kill drill** — a planned failure, such as a SIGKILL on a replica under load, with the checks written down before it
  runs (lessons 34 and 35).
- **Budget cap** — a limit on model calls and spend per turn, checked before the next call (lessons 06 and 37).
- **Spend cap** — the dollar limit inside the budget cap; the turn stops when the next call would pass it (lessons 06 and 37).
- **Idempotency key** — a key sent with a write, made from the request, so a repeat returns the first answer (lessons 32
  and 34).
- **Audit log** — a record of who did what on whose behalf, with the reviewer of each money move (lesson 31).
- **Span** — one timed step of a run in a trace, such as a model call or a tool call (lesson 29).
- **Trace** — the tree of spans for one run, from the first step to the last, so one turn can be followed end to end (lesson 29).